# Qwen3-TTS Fine-tuning 완전 초보자 가이드

**이 노트북은 TTS를 전혀 모르는 분도 이해할 수 있도록 작성되었습니다.**  
여기서는 파인튜닝에 대한 자세한 내용을 배웁니다. Qwen3-TTS에 대해 자세히 알고 싶은 분들은 https://wikidocs.net/325432 링크를 참고하시면 됩니다. 


## TTS란 무엇인가요?

**TTS(Text-to-Speech)** 는 텍스트를 음성으로 변환하는 기술입니다. `"안녕하세요"`라는 텍스트를 입력하면, 실제 사람이 말하는 것처럼 자연스러운 음성 파일이 생성됩니다. 마치 성우가 대본을 받아 자기 목소리로 읽어서 녹음 파일을 만들어내는 과정과 비슷합니다.



### 핵심 개념 사전

이 노트북에서 반복적으로 등장하는 용어들을 먼저 정리합니다.

#### Speaker Embedding (화자 임베딩)

사람마다 지문이 다르듯 목소리에도 고유한 특징이 있습니다. Speaker Embedding은 한 사람의 목소리 특징을 1024개의 숫자로 요약한 벡터입니다. 예를 들어 `[0.12, -0.34, 0.56, ...]`처럼 생긴 이 벡터 안에 "낮은 톤", "허스키한 음색", "빠른 말투" 같은 정보가 담겨 있으며, 참조 음성(Reference Audio)에서 자동으로 추출됩니다.

#### Codec (코덱)

음성은 원래 연속적인 파형(wave)인데, 이 상태로는 AI가 다루기 어렵습니다. 코덱은 이 연속적인 파형을 142, 87, 523처럼 띄엄띄엄 끊어진 정수 번호들로 바꿔주는 역할을 합니다. 우리가 글을 쓸 때 연속적인 생각을 "단어"라는 덩어리로 쪼개는 것처럼, 코덱은 연속적인 음성을 "코덱 토큰"이라는 숫자 덩어리로 쪼개는 것입니다. Qwen3-TTS는 **16개의 코드북(Sub-codebook)** 을 사용하는데, 코드북 0이 발음과 억양 같은 핵심 정보를 담당하고, 코드북 1~15가 음질과 세부 특성을 보완합니다.

  
  Before Codec (Tacotron 2, FastSpeech 등)
  
  텍스트 → Encoder → Mel-Spectrogram → Vocoder → 음성 파형
                    (80-dim 연속 벡터의 시퀀스)    (HiFi-GAN 등)

  Mel Spectrogram 은 연속값(continuous) 입니다:
  - 매 프레임마다 80차원 실수 벡터
  - 예: [0.234, -0.561, 1.023, ..., -0.117] (80개의 float)
  - 모델은 이 실수값을 회귀(regression) 로 예측해야 함

  문제점:
  - 연속 회귀는 LLM 의 "다음 토큰 예측" 패러다임과 안 맞음
  - Vocoder 라는 별도 단계가 필요 (2-stage 시스템)
  - 사전학습 음성 데이터를 자기지도 학습으로 활용하기 어려움
  - 음색을 미세 조정하려면 음향 모델 + vocoder 둘 다 손봐야 함

  With Codec (Qwen3-TTS, VALL-E, Bark 등)

  텍스트 → LLM (Talker) → Codec Token → Codec Decoder → 음성 파형
                         (정수 토큰의 시퀀스)            (사전학습된 디코더)

  Codec Token 은 이산값(discrete) 입니다:
  - 매 프레임마다 16개의 정수 (예: [142, 87, 523, ...])
  - 모델은 "다음 토큰" 을 분류(classification) 로 예측 — 언어 모델과 동일한 방식
  - Qwen3-TTS 의 talker 가 LLM 으로 만들어진 이유

  왜 코덱이 게임체인저였나?

  이게 핵심입니다 — 코덱의 등장으로 "음성도 언어처럼 다룰 수 있게" 됐어요.

| 기능 | Mel 기반 TTS | Codec 기반 TTS |
|---|---|---|
| 모델 구조 | Encoder-Decoder (CNN/RNN) | Transformer LM (GPT 같은) |
| 학습 목표 | 연속값 MSE 회귀 | 이산 토큰 분류 (cross-entropy) |
| 스케일링 | 100M~500M params 한계 | 수십억~수백억 params 가능 |
| 자기지도 사전학습 | 어려움 (mel 자체가 lossy) | 음성 데이터로 LM 사전학습 가능 |
| In-context learning | 거의 불가능 | 3초 샘플로 zero-shot voice clone |
| 다국어/다화자 | 별도 모델 필요 | 하나의 모델이 모두 처리 |
| Vocoder 의존 | HiFi-GAN 등 따로 학습/관리 | 코덱 디코더가 함께 학습됨 |


#### Mel Spectrogram (멜 스펙트로그램)

Mel Spectrogram은 음성을 시간-주파수 평면에 시각화한 것으로, 음성의 사진이라고 볼 수 있습니다. 가로축은 시간, 세로축은 주파수(128차원), 밝기는 에너지를 나타냅니다. Speaker Encoder가 이 이미지를 분석하여 화자의 목소리 특징을 추출합니다.
https://docs.pytorch.org/audio/0.11.0/transforms.html

## Qwen3-TTS 아키텍처 전체 흐름도

아래 다이어그램은 Fine-tuning 과정에서 데이터가 모델 내부를 어떻게 흘러가는지를 보여줍니다.

```
=================================================================
             Qwen3-TTS 학습 파이프라인
=================================================================

  [입력 데이터]
  ┌─────────────────────────────────────────────────────────────┐
  │  텍스트: "안녕하세요"                                           │
  │  참조 음성: reference.wav (화자의 샘플 음성)                      │
  │  정답 코덱: [142, 87, 523, ...] x 16개 코드북                   │
  └─────────────────────────────────────────────────────────────┘
           │                    │              │
           v                    v              │
     [Text Embedding]   [Speaker Encoder]      │
     텍스트 → 벡터       참조음성 → 목소리 지문        │
            │                   │              │
            v                   v              │
     ┌──────────────────────────────────┐      │
     │    임베딩 합산 (Position 6 주입)     │      │
     │    text + codec + speaker        │      │
     └──────────────┬───────────────────┘      │
                    │                          │
                    v                          v
              [Talker LLM]                    [정답 코덱]
              코덱 토큰 예측                       │
                    │                          │
                    v                          v
              ┌─────────────────────────────────────┐
              │          Loss 계산                   │
              │   예측 코덱 vs 정답 코덱 비교            │
              └─────────────────────────────────────┘
                              │
                              v
                        모델 가중치 업데이트
```


## 0단계: 학습 데이터 준비 (HuggingFace에서 다운로드)

이 튜토리얼에서는 [daje/korean-tts-training](https://huggingface.co/datasets/daje/korean-tts-training) 데이터셋을 사용합니다.

이 데이터셋의 특징은 다음과 같습니다.

| 항목 | 내용 |
| --- | --- |
| 문장 수 | 120개 |
| 화자 | 단일 화자 (Gemini Zephyr 음성) |
| 포맷 | WAV (24kHz, 16bit, Mono) |
| 라이선스 | CC-BY-4.0 |
| 카테고리 | 발음(숫자, 겹받침, 음운변동 등), 감정(긍정, 부정, 복합) |

데이터 준비는 3단계로 진행됩니다.

1. **HuggingFace에서 다운로드**: `datasets` 라이브러리로 오디오와 텍스트를 가져옵니다.
2. **WAV 파일 저장**: 오디오를 로컬 폴더에 개별 WAV 파일로 저장합니다.
3. **Audio Codes 생성**: Qwen3-TTS Tokenizer로 음성을 코덱 토큰으로 변환합니다.


In [ ]:
# ============================================================
# 0단계: HuggingFace 데이터셋 다운로드 → JSONL 변환
# ============================================================
# 이 셀은 최초 1회만 실행하면 됩니다.
# 이미 JSONL 파일이 있다면 건너뛰세요.
# ============================================================

import os
import json
import soundfile as sf
from datasets import load_dataset

# --- 저장 경로 설정 ---
# BASE_DIR: 이 노트북이 위치한 디렉토리
BASE_DIR  = os.getcwd()
DATA_DIR  = os.path.join(BASE_DIR, "data")          # 데이터 저장 폴더
AUDIO_DIR = os.path.join(DATA_DIR, "audio")         # WAV 파일 저장 폴더
os.makedirs(AUDIO_DIR, exist_ok=True)

# 1) HuggingFace에서 데이터셋 다운로드
print("HuggingFace에서 데이터셋을 다운로드합니다...")
ds = load_dataset("daje/korean-tts-training", split="train")
print(f"다운로드 완료: {len(ds)}개 샘플")

# 2) WAV 파일 저장 + 초기 JSONL 생성
#    ref_audio는 첫 번째 샘플의 음성을 사용합니다.
#    (단일 화자 데이터셋이므로 어떤 파일이든 참조 음성으로 사용 가능)
ref_audio_path = os.path.join(AUDIO_DIR, f"audio_{ds[0]['id']}.wav")

raw_jsonl_path = os.path.join(DATA_DIR, "train_raw.jsonl")  # audio_codes 없는 중간 파일

with open(raw_jsonl_path, "w", encoding="utf-8") as f:
    for sample in ds:
        audio_data = sample["audio"]  # {"array": [...], "sampling_rate": 24000}
        wav_path = os.path.join(AUDIO_DIR, f"audio_{sample['id']}.wav")

        # WAV 파일 저장
        sf.write(wav_path, audio_data["array"], audio_data["sampling_rate"])

        # JSONL 한 줄 기록
        line = {
            "audio": wav_path,
            "text": sample["text"],
            "ref_audio": ref_audio_path,
        }
        f.write(json.dumps(line, ensure_ascii=False) + "\n")

print(f"WAV 파일 {len(ds)}개 저장 완료: {AUDIO_DIR}")
print(f"중간 JSONL 생성 완료: {raw_jsonl_path}")

In [2]:
# ============================================================
# audio_codes 생성 (Qwen3-TTS Tokenizer 사용)
# ============================================================
# prepare_data.py의 전체 코드입니다.
# 음성 파일을 코덱 토큰으로 변환합니다.
# GPU가 필요하며, 120개 샘플 기준 약 1~2분 소요됩니다.
# ============================================================

from qwen_tts import Qwen3TTSTokenizer

# --- 설정 ---
DEVICE = "cuda:0"
# BASE_DIR: 이 노트북이 위치한 디렉토리 (어떤 서버에서도 동작)
BASE_DIR = os.getcwd()

# TOKENIZER_PATH: HuggingFace Hub ID 사용 — 첫 실행 시 자동 다운로드됨
#   로컬에 이미 다운받은 폴더가 있다면 그 경로로 바꿔도 OK
TOKENIZER_PATH = "Qwen/Qwen3-TTS-Tokenizer-12Hz"
INPUT_JSONL  = os.path.join(BASE_DIR, "data", "train_raw.jsonl")
OUTPUT_JSONL = os.path.join(BASE_DIR, "data", "train_with_codes.jsonl")
BATCH_INFER_NUM = 32  # 한 번에 처리할 오디오 파일 수

# 1) Tokenizer 로드
#    Qwen3-TTS Tokenizer는 음성 파형을 코덱 토큰으로 변환하는 인코더입니다.
#    12Hz는 1초당 12개의 코덱 프레임을 생성한다는 뜻입니다.
#    예: 3초 음성 → 36개 프레임, 각 프레임은 16개 코드북 값을 가짐
tokenizer_12hz = Qwen3TTSTokenizer.from_pretrained(
    TOKENIZER_PATH,
    device_map=DEVICE,
)
print(f"Tokenizer 로드 완료: {TOKENIZER_PATH}")

# 2) 입력 JSONL 읽기 (0단계에서 생성한 파일)
total_lines = open(INPUT_JSONL).readlines()
total_lines = [json.loads(line.strip()) for line in total_lines]
print(f"입력 데이터: {len(total_lines)}개 샘플")

# 3) 배치 단위로 audio_codes 생성
#    한 번에 BATCH_INFER_NUM개씩 묶어서 처리하면 GPU를 효율적으로 활용할 수 있습니다.
#    tokenizer.encode()는 WAV 파일 경로 리스트를 받아서
#    각 음성에 대한 audio_codes (shape: [16, 프레임수])를 반환합니다.
final_lines = []
batch_lines = []
batch_audios = []

for i, line in enumerate(total_lines):
    batch_lines.append(line)
    batch_audios.append(line["audio"])

    if len(batch_lines) >= BATCH_INFER_NUM:
        enc_res = tokenizer_12hz.encode(batch_audios)
        for code, bl in zip(enc_res.audio_codes, batch_lines):
            bl["audio_codes"] = code.cpu().tolist()
            final_lines.append(bl)
        print(f"  처리 완료: {len(final_lines)}/{len(total_lines)}")
        batch_lines.clear()
        batch_audios.clear()

# 남은 샘플 처리
if len(batch_audios) > 0:
    enc_res = tokenizer_12hz.encode(batch_audios)
    for code, bl in zip(enc_res.audio_codes, batch_lines):
        bl["audio_codes"] = code.cpu().tolist()
        final_lines.append(bl)
    batch_lines.clear()
    batch_audios.clear()

# 4) 결과 저장
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for line in final_lines:
        f.write(json.dumps(line, ensure_ascii=False) + "\n")

# 5) 결과 확인
first = final_lines[0]
print(f"총 {len(final_lines)}개 샘플 변환 완료")
print(f"audio_codes 코드북 수: {len(first['audio_codes'])}")
print(f"첫 코드북 길이 (프레임 수): {len(first['audio_codes'][0])}")
print(f"텍스트: {first['text']}")
print(f"✅ 데이터 준비 완료! 1단계로 넘어가세요.")

# Tokenizer GPU 메모리 해제 (학습에 필요한 메모리 확보)
del tokenizer_12hz
import torch
torch.cuda.empty_cache()
print("Tokenizer 메모리 해제 완료")


Tokenizer 로드 완료: Qwen/Qwen3-TTS-Tokenizer-12Hz
입력 데이터: 120개 샘플
  처리 완료: 32/120
  처리 완료: 64/120
  처리 완료: 96/120
총 120개 샘플 변환 완료
audio_codes 코드북 수: 121
첫 코드북 길이 (프레임 수): 16
텍스트: 고객센터 번호는 1588-3920이고, 팩스는 02-555-7812입니다.
✅ 데이터 준비 완료! 1단계로 넘어가세요.
Tokenizer 메모리 해제 완료


## 1단계: 설정

학습에 필요한 모든 설정값을 아래 셀에서 관리합니다. 이 셀의 값만 수정하면 나머지 코드는 그대로 실행할 수 있습니다.


In [ ]:
# BASE_DIR: 이 노트북이 위치한 디렉토리 (서버 이식성 확보)
BASE_DIR = os.getcwd()

# INIT_MODEL_PATH: HuggingFace Hub ID 사용 — 첫 실행 시 자동 다운로드됨
#   - 0.6B: "Qwen/Qwen3-TTS-12Hz-0.6B-Base" (이 워크숍 기본값)
#   - 1.7B: "Qwen/Qwen3-TTS-12Hz-1.7B-Base" (더 큰 모델)
#   이미 로컬 다운로드한 폴더가 있다면 그 절대 경로로 바꿔도 됩니다.
INIT_MODEL_PATH   = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"

# OUTPUT_MODEL_PATH: Fine-tuning 완료된 모델을 저장할 폴더 경로
#   학습 중 매 epoch마다 checkpoint-epoch-N/ 하위 폴더가 생성됩니다.
OUTPUT_MODEL_PATH = os.path.join(BASE_DIR, "outputs", "finetuned_korean_tts")

# TRAIN_JSONL: 학습 데이터 파일 경로 (JSONL 형식)
#   각 줄은 {"audio": "경로", "text": "대사", "audio_codes": [...], "ref_audio": "경로"} 형태
TRAIN_JSONL       = os.path.join(BASE_DIR, "data", "train_with_codes.jsonl")

# SPEAKER_NAME: 학습시킬 화자의 이름 (영문, 저장 시 식별자로 사용)
#   추론 시 model.generate_custom_voice(text=..., speaker=SPEAKER_NAME) 으로 호출합니다.
SPEAKER_NAME      = "korean_tts"


# --- 학습 하이퍼파라미터 ---

# BATCH_SIZE: 한 번에 몇 개의 샘플을 동시에 학습하는지
#   - 크게 하면: 학습이 안정적이지만, GPU 메모리를 많이 사용
#   - 작게 하면: 메모리 절약되지만, 학습이 불안정할 수 있음
#   - 보통 2~4 사이로 설정 (GPU 메모리에 따라 조절)
BATCH_SIZE        = 2

# LEARNING_RATE: 모델이 한 번의 업데이트에서 얼마나 변화하는지 (학습률)
#   - 비유: "한 걸음의 크기"
#   - 크게 하면: 빠르게 학습하지만, 최적점을 지나칠 수 있음 (발산)
#   - 작게 하면: 안정적이지만, 학습이 너무 느림
#
#   [중요] 화자 클로닝에서 LR이 너무 크면(예: 2e-5) autoregressive talker의
#   EOS(End-Of-Sentence) 분포가 무너져, 추론 시 max_new_tokens까지 멈추지 못하고
#   수백 초짜리 잡음을 생성하는 "EOS 붕괴" 현상이 일어납니다.
#   본 노트북 작성자는 120 샘플 데이터에 대해 2e-7로 학습했을 때
#   EOS 분포가 보존되고 30개 테스트 문장 모두 정상 길이(평균 4.05s)로 생성되었습니다.
#   데이터가 더 많다면(>500 샘플) 2e-7 정도까지도 시도해볼 수 있습니다.
LEARNING_RATE     = 2e-7

# NUM_EPOCHS: 전체 학습 데이터를 몇 번 반복 학습할지
#   - 1 Epoch = 모든 데이터를 1번 훑음
#   - 너무 적으면: 학습 부족 (음성 품질 낮음)
#   - 너무 많으면: 과적합 (학습 데이터만 잘 따라하고 새로운 텍스트에 약해짐)
#   - 데이터가 적으면(10~50개) 3~10 Epoch, 많으면(500+) 1~3 Epoch
#
#   [기록] 120 샘플 + LR 2e-7 조합에서 8 epoch(=480 step)이 sweet spot이었습니다.
#   필요하면 recommend_epochs.py 로 데이터 수에 맞는 권장값을 받을 수 있습니다.
NUM_EPOCHS        = 8

# GRAD_ACCUM_STEPS: Gradient Accumulation Steps (그래디언트 누적 스텝)
#   - 작은 배치를 여러 번 모아서 큰 배치처럼 학습하는 기법
#   - 실효 배치 크기 = BATCH_SIZE x GRAD_ACCUM_STEPS = 2 x 4 = 8
#   - GPU 메모리가 부족할 때, 배치 크기를 줄이고 이 값을 늘리면
#     큰 배치로 학습한 것과 비슷한 효과를 낼 수 있음
GRAD_ACCUM_STEPS  = 4


## 2단계: Import 및 초기화

학습에 필요한 라이브러리들을 불러옵니다. 각 라이브러리가 담당하는 역할은 다음과 같습니다.

| 라이브러리 | 역할 |
|-----------|------|
| `torch` | 딥러닝의 기본 프레임워크로, 텐서 연산과 자동 미분을 처리합니다 |
| `accelerate` | 멀티 GPU 분산 학습과 혼합 정밀도 학습을 자동으로 관리합니다 |
| `transformers` | 모델 설정(config.json)을 로드하는 데 사용합니다 |
| `safetensors` | 모델 가중치를 안전하게 저장하는 파일 형식입니다 |
| `TTSDataset` | 학습 데이터를 PyTorch가 이해하는 형태로 변환하는 커스텀 클래스입니다 |
| `Qwen3TTSModel` | Qwen3-TTS 모델을 로드하고 추론하는 래퍼 클래스입니다 |


In [4]:
import json           # JSON 파일 읽기/쓰기 (설정 파일, 학습 데이터)
import os             # 파일 경로 조작 (os.path.join 등)
import shutil         # 폴더 복사 (체크포인트 저장 시 사용)
import sys            # Python 경로 설정 (커스텀 모듈 import용)

import torch                          # PyTorch 핵심 라이브러리
from accelerate import Accelerator    # 분산 학습 + 혼합 정밀도 관리
from safetensors.torch import save_file  # 모델 가중치를 .safetensors 형식으로 저장
from torch.optim import AdamW         # 옵티마이저 (가중치 업데이트 알고리즘)
from torch.utils.data import DataLoader  # 데이터를 배치 단위로 공급
from transformers import AutoConfig   # 모델 설정 파일(config.json) 자동 로드

# finetuning 디렉토리를 Python 경로에 추가
# 이렇게 해야 같은 폴더에 있는 dataset.py를 import할 수 있습니다
FINETUNE_DIR = os.path.dirname(os.path.abspath("__file__"))
if os.getcwd() not in sys.path:
    # dataset.py 가 이 노트북과 같은 폴더에 있어야 import 됨.
# Jupyter 는 노트북이 있는 폴더를 cwd 로 설정하므로 os.getcwd() 사용.
    sys.path.insert(0, os.getcwd())

# 커스텀 모듈 import
from dataset import TTSDataset                          # 학습 데이터셋 클래스
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel  # TTS 모델 래퍼

print("Import 완료")

Import 완료


## 3단계: 모델 로드

사전학습된 Qwen3-TTS 모델을 GPU에 올립니다.

**Accelerator**는 학습 과정 전체를 관리하는 도구입니다. `gradient_accumulation_steps=4`는 작은 배치를 4번 모아서 한 번에 가중치를 업데이트하겠다는 뜻이고, `mixed_precision="bf16"`은 BFloat16 혼합 정밀도를 사용하여 메모리를 절약하면서 학습 속도를 높입니다.

**Qwen3TTSModel.from_pretrained()** 는 사전학습된 모델 파일을 읽어서 GPU에 올립니다. `flash_attention_2` 옵션을 사용하면 어텐션 연산 속도가 크게 향상됩니다.

**AutoConfig.from_pretrained()** 는 모델의 설정 정보(config.json)를 읽어옵니다. 여기에 모델의 차원 크기, 특수 토큰 ID 등 학습에 필요한 정보가 담겨 있습니다.


In [ ]:
# Accelerator 생성: 학습 과정 전체를 관리하는 "지휘자" 역할
accelerator = Accelerator(
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,  # 그래디언트 누적 횟수
    mixed_precision="bf16",                        # BFloat16 혼합 정밀도
    log_with="tensorboard"                         # TensorBoard 로깅
)

# 사전학습된 TTS 모델 로드
# 이 모델 안에는 여러 컴포넌트가 들어있습니다:
#   - model.speaker_encoder: 참조 음성에서 화자 임베딩을 추출하는 네트워크
#   - model.talker: 텍스트+화자정보를 받아 코덱을 예측하는 LLM
#   - model.talker.model.text_embedding: 텍스트 토큰을 벡터로 변환
#   - model.talker.model.codec_embedding: 코덱 토큰을 벡터로 변환
#   - model.talker.code_predictor: Sub-codebook(1~15)을 예측하는 네트워크
qwen3tts = Qwen3TTSModel.from_pretrained(
    INIT_MODEL_PATH,                    # 모델 폴더 경로
    torch_dtype=torch.bfloat16,         # BF16으로 로드 (메모리 절약)
    attn_implementation="flash_attention_2",  # FlashAttention2 (속도 향상)
)

# 모델 설정(config) 로드: 모델의 구조 정보가 담겨있음
config = AutoConfig.from_pretrained(INIT_MODEL_PATH)

print(f"모델 로드 완료: {INIT_MODEL_PATH}")

## 4단계: 0.6B / 1.7B 모델 자동 감지

로드한 모델이 0.6B인지 1.7B인지를 자동으로 판별합니다. 핵심은 `text_hidden_size`와 `hidden_size`를 비교하는 것입니다.

1.7B 모델은 두 값이 동일하지만(2048), 0.6B 모델은 텍스트 차원이 2048이고 음성 차원이 1024로 서로 다릅니다. 값이 다르면 `text_projection` 레이어를 적용하여 차원을 맞춰야 합니다. 이 감지를 건너뛰면 0.6B에서는 차원 불일치(dimension mismatch) 오류가 발생하고, 1.7B에서 불필요하게 적용하면 잘못된 결과가 나옵니다.


In [6]:
# config에서 talker 설정을 가져옴
# talker_config 안에 모델의 차원 정보가 들어있음
talker_config = config.talker_config if hasattr(config, 'talker_config') else config

# text_hidden_size: 텍스트 임베딩의 차원 (보통 2048)
text_hidden_size = getattr(talker_config, 'text_hidden_size', 2048)

# hidden_size: Talker LLM 내부의 차원
#   - 0.6B: 1024 (텍스트 임베딩 2048과 다름! -> projection 필요)
#   - 1.7B: 2048 (텍스트 임베딩과 같음 -> projection 불필요)
hidden_size = getattr(talker_config, 'hidden_size', 2048)

# 두 차원이 다르면 projection이 필요한 0.6B 모델
needs_projection = (text_hidden_size != hidden_size)

if needs_projection:
    print(f"[INFO] 0.6B 모델 감지!")
    print(f"  - text_hidden_size = {text_hidden_size} (텍스트 임베딩 차원)")
    print(f"  - hidden_size = {hidden_size} (Talker 내부 차원)")
    print(f"  - 차원이 다르므로 text_projection을 적용합니다")
    print(f"  - {text_hidden_size} -> text_projection -> {hidden_size}")
else:
    print(f"[INFO] 1.7B 모델 감지!")
    print(f"  - 두 차원이 모두 {hidden_size}로 동일")
    print(f"  - text_projection이 필요 없습니다")

[INFO] 0.6B 모델 감지!
  - text_hidden_size = 2048 (텍스트 임베딩 차원)
  - hidden_size = 1024 (Talker 내부 차원)
  - 차원이 다르므로 text_projection을 적용합니다
  - 2048 -> text_projection -> 1024


### 🎧 학습 데이터 미리듣기 — "어떤 목소리를 학습할까?"

본격적인 학습에 들어가기 전에, 우리가 학습시킬 목소리(타깃 화자)가 어떻게 들리는지 먼저 확인해보세요. 나중에 fine-tuning 결과와 비교하면 "학습이 잘 되었는지"를 귀로 직접 검증할 수 있습니다.

In [7]:
# ============================================================
# 학습 데이터 미리듣기 (다운로드된 첫 3개 샘플)
# ============================================================
from IPython.display import Audio, display
import json

raw_jsonl = os.path.join(DATA_DIR, "train_raw.jsonl")
with open(raw_jsonl) as f:
    preview_samples = [json.loads(line) for line in f.readlines()[:2]]

print(f"학습 데이터 (목표 화자) 샘플 {len(preview_samples)}개:")
print(f"   → 이 목소리를 fine-tuning 으로 학습할 것입니다.\n")

for i, s in enumerate(preview_samples, 1):
    print(f"[{i}] {s['text']}")
    display(Audio(s['audio'], rate=24000))


학습 데이터 (목표 화자) 샘플 2개:
   → 이 목소리를 fine-tuning 으로 학습할 것입니다.

[1] 고객센터 번호는 1588-3920이고, 팩스는 02-555-7812입니다.


[2] 이번 달 월세 850,000원에 관리비 123,000원을 더하면 973,000원이에요.


### 🎤 학습 전 BASE 모델 — Qwen 의 한국어 화자 sohee 로 발성

Qwen3-TTS 의 **CustomVoice 모델** 에는 9명의 사전등록 화자가 있는데, 그 중 **`sohee`** 가 한국어 여성 화자입니다. 이 화자로 같은 문장을 발음시켜서 우리 학습 데이터의 목표 화자와 비교해보세요.

| 청취 비교 | 입력 | 모델 |
|---|---|---|
| 위 샘플 [1] | (학습 데이터 원본) | — |
| 이 셀 출력 | `text + speaker="sohee"` | **Qwen3-TTS-CustomVoice (학습 안 함)** |
| 학습 후 (Cell 32) | `text + speaker="korean_tts"` | **Fine-tuned by us** |

같은 한국어 여성 발화여도 **sohee 와 우리 목표 화자는 다른 사람** 입니다. Fine-tuning 후 결과가 sohee 보다 우리 목표 화자에 훨씬 가까우면 학습이 성공한 것입니다.

> ⚡ 이 셀은 `generate_voice_clone` 보다 훨씬 빠릅니다 — 사전등록 화자 토큰 1개만 쓰므로 ICL 컨텍스트가 없거든요.

In [8]:
# ============================================================
# CustomVoice 0.6B 모델로 sohee 화자 한국어 발성 테스트
# (BASE 모델 직접 호출 대신, 9명의 화자가 미리 학습된 CustomVoice 활용)
# ============================================================
import torch
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

raw_jsonl = os.path.join(DATA_DIR, "train_raw.jsonl")
with open(raw_jsonl) as f:
    first = json.loads(f.readline())
test_text = first["text"]

print(f"테스트 텍스트 (위 샘플 [1] 과 동일):")
print(f"  {test_text}\n")

# CustomVoice 0.6B 임시 로드 (HF 캐시 첫 실행 시 ~2GB 다운로드)
CUSTOMVOICE_PATH = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"
print(f"CustomVoice 모델 로드 중: {CUSTOMVOICE_PATH}")
print(f"  (첫 실행 시 ~2GB HF 다운로드, 이후엔 캐시에서 즉시 로드)\n")

cv_model = Qwen3TTSModel.from_pretrained(
    CUSTOMVOICE_PATH,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

speakers = cv_model.get_supported_speakers()
print(f"CustomVoice 모델 사전등록 화자: {speakers}")
print(f"사용 화자: sohee (한국어 여성)\n")

base_audio_path = os.path.join(BASE_DIR, "_base_model_korean_test.wav")

try:
    wavs, sr = cv_model.generate_custom_voice(text=test_text, speaker="sohee")
    sf.write(base_audio_path, wavs[0], sr)
    print(f"✅ sohee 발성 생성 성공")
    print(f"   재생 시간: {len(wavs[0])/sr:.2f}초")
    print(f"   저장 위치: {base_audio_path}\n")
    print(f"   ↑ 위 샘플 [1] (우리 학습 데이터 화자) 와 비교해 들어보세요.")
    print(f"      같은 한국어 여성 화자지만 sohee 와 우리 목표 화자는 다른 사람.")
    print(f"      Fine-tuning 후 결과 (Cell 32-33) 는 [1] 에 훨씬 가까울 것입니다.\n")
    display(Audio(wavs[0], rate=sr))
except Exception as e:
    print(f"⚠️ sohee 생성 실패: {e}")
finally:
    # 학습용 qwen3tts 와 메모리 충돌 방지를 위해 즉시 해제
    del cv_model
    torch.cuda.empty_cache()
    print(f"   CustomVoice 모델 메모리 해제 완료 (학습에 영향 없음)")


테스트 텍스트 (위 샘플 [1] 과 동일):
  고객센터 번호는 1588-3920이고, 팩스는 02-555-7812입니다.

CustomVoice 모델 로드 중: Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice
  (첫 실행 시 ~2GB HF 다운로드, 이후엔 캐시에서 즉시 로드)



Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 3677.60it/s]


CustomVoice 모델 사전등록 화자: ['aiden', 'dylan', 'eric', 'ono_anna', 'ryan', 'serena', 'sohee', 'uncle_fu', 'vivian']
사용 화자: sohee (한국어 여성)



Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ sohee 발성 생성 성공
   재생 시간: 9.50초
   저장 위치: /dataset/research/temp/AI_workshop/chapter5-2_TTS/_base_model_korean_test.wav

   ↑ 위 샘플 [1] (우리 학습 데이터 화자) 와 비교해 들어보세요.
      같은 한국어 여성 화자지만 sohee 와 우리 목표 화자는 다른 사람.
      Fine-tuning 후 결과 (Cell 32-33) 는 [1] 에 훨씬 가까울 것입니다.



   CustomVoice 모델 메모리 해제 완료 (학습에 영향 없음)


## 5단계: 학습 데이터 로드

0단계에서 준비한 JSONL 파일을 읽어서, PyTorch DataLoader로 배치 단위로 공급할 준비를 합니다.

`daje/korean-tts-training` 데이터셋은 120개 샘플이므로, `BATCH_SIZE=2`일 때 60개의 배치가 만들어집니다.

**DataLoader**는 전체 데이터를 지정한 배치 크기로 잘라서 하나씩 공급하는 역할을 합니다. `shuffle=True`를 지정하면 에폭마다 데이터 순서가 무작위로 섞여서 학습이 더 안정적으로 진행됩니다.

In [9]:
# JSONL 파일을 한 줄씩 읽어서 리스트로 변환
# 각 줄을 JSON으로 파싱하면 Python 딕셔너리가 됨
train_data = open(TRAIN_JSONL).readlines()
train_data = [json.loads(line) for line in train_data]

# TTSDataset: 원시 데이터를 모델이 이해하는 텐서(Tensor)로 변환하는 클래스
# - 텍스트를 토큰 ID로 변환
# - 참조 음성을 Mel Spectrogram으로 변환
# - 코덱 데이터를 텐서로 변환
dataset = TTSDataset(train_data, qwen3tts.processor, config)

# DataLoader: 데이터셋에서 배치 단위로 데이터를 꺼내주는 도구
# - batch_size: 한 번에 몇 개 샘플을 묶을지
# - shuffle=True: 매 에폭마다 데이터 순서를 섞음 (과적합 방지)
# - collate_fn: 서로 다른 길이의 샘플들을 패딩해서 같은 크기로 맞춤
train_dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=dataset.collate_fn
)

# 데이터 정보 출력
print(f"학습 데이터: {len(train_data)}개 샘플")
print(f"배치 크기: {BATCH_SIZE}")
print(f"총 배치 수: {len(train_dataloader)} (= {len(train_data)} / {BATCH_SIZE}, 올림)")
print(f"실효 배치 크기: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")

학습 데이터: 120개 샘플
배치 크기: 2
총 배치 수: 60 (= 120 / 2, 올림)
실효 배치 크기: 2 x 4 = 8


## 6단계: 옵티마이저 및 Accelerator 준비

모델이 예측한 결과와 정답 사이의 차이(Loss)를 계산하면, 그 차이를 줄이는 방향으로 모델의 가중치를 조금씩 수정해야 합니다. 이 수정 방법을 결정하는 것이 **옵티마이저(Optimizer)** 입니다.

이 노트북에서는 **AdamW** 옵티마이저를 사용합니다. AdamW는 Adam 옵티마이저의 개선 버전으로, 가중치가 지나치게 커지는 것을 방지하는 가중치 감쇠(Weight Decay)를 올바르게 적용합니다.

`accelerator.prepare()`를 호출하면 모델, 옵티마이저, 데이터로더가 Accelerator에 등록되어 GPU 배치, 분산 처리, BF16 혼합 정밀도, 그래디언트 누적이 모두 자동으로 관리됩니다.


In [10]:
# AdamW 옵티마이저 생성
# - qwen3tts.model.parameters(): 모델의 모든 학습 가능한 가중치
# - lr: Learning Rate (학습률)
# - weight_decay: L2 정규화 강도 (과적합 방지)
optimizer = AdamW(qwen3tts.model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Accelerator에 모델, 옵티마이저, 데이터로더를 등록
# 이 시점부터 model, optimizer, train_dataloader는 Accelerator가 관리합니다
# (GPU 배치, 혼합 정밀도, 그래디언트 누적 등 자동 처리)
model, optimizer, train_dataloader = accelerator.prepare(
    qwen3tts.model, optimizer, train_dataloader
)

print("Accelerator 준비 완료")
print(f"모델 디바이스: {model.device}")
print(f"학습률: {LEARNING_RATE}")
print(f"Weight Decay: 0.01")

Accelerator 준비 완료
모델 디바이스: cuda:0
학습률: 2e-07
Weight Decay: 0.01


## 7단계: 학습 루프

이제 실제로 모델을 학습시킵니다. 이 단계가 Fine-tuning의 핵심이며, 아래 코드 셀 안에서 여러 작업이 순서대로 진행됩니다.


[7-1] 배치 데이터 언패킹 : DataLoader가 공급한 배치(batch)에서 각 텐서를 꺼냅니다. 배치에는 텍스트 토큰, 코덱 토큰, 참조 음성의 Mel Spectrogram, 각종 마스크 등이 딕셔너리 형태로 들어 있습니다.

[7-2] Speaker Embedding 추출 : `model.speaker_encoder(ref_mels)`는 참조 음성의 Mel Spectrogram을 입력받아 화자의 목소리 특징을 1024차원 벡터로 추출합니다. `.detach()`를 붙여서 이 벡터에는 그래디언트가 흐르지 않도록 합니다. Speaker Encoder 자체는 학습시키지 않고, 이미 잘 학습된 상태를 그대로 활용하기 위함입니다.

[7-3] Text Embedding 생성 : 텍스트 토큰을 벡터로 변환합니다. 0.6B 모델인 경우 `text_projection`을 추가로 적용하여 2048차원을 1024차원으로 줄이고, 1.7B 모델은 차원이 이미 같으므로 바로 사용합니다.

[7-4] Codec Embedding + 화자 주입 : 코덱 토큰도 벡터로 변환한 뒤, **시퀀스의 6번 위치에 Speaker Embedding을 주입**합니다. 이 6번 위치는 모델이 "누구의 목소리로 말할지"를 인식하는 전용 슬롯으로, 학습 시에는 참조 음성에서 추출한 임베딩을 넣고, 추론 시에는 저장된 화자 임베딩을 넣습니다.

[7-5] 임베딩 합산 : Text Embedding과 Codec Embedding을 더합니다. 두 정보를 합산하면 모델이 "어떤 텍스트를 어떤 화자의 목소리로 읽을지"를 동시에 파악할 수 있게 됩니다.

[7-6] Sub-codebook 임베딩 추가 : 코드북 1~15의 임베딩도 합산합니다. 코드북 0만으로도 기본적인 음성을 표현할 수 있지만, 나머지 15개 코드북이 음질과 세부 뉘앙스를 보완합니다.

[7-7] Forward Pass (Talker) : 합산된 임베딩을 Talker LLM에 입력하면 다음 코덱 토큰을 예측합니다. `output_hidden_states=True`를 지정하면 중간 은닉 상태도 함께 반환되는데, 이 값은 sub-talker 학습에 사용됩니다.

[7-8] Sub-talker Loss 계산 : Talker의 은닉 상태를 이용하여 코드북 1~15의 예측 정확도도 함께 계산합니다. 이 과정을 통해 코드북 0뿐만 아니라 모든 코드북의 품질이 동시에 향상됩니다.

[7-9] 최종 Loss 합산 및 역전파 : 메인 Loss(코드북 0 예측)에 sub-talker Loss의 30%를 더한 값이 최종 Loss입니다. `accelerator.backward(loss)`로 역전파를 수행하고, 그래디언트 클리핑(최대 1.0)을 적용한 뒤 옵티마이저가 가중치를 업데이트합니다.


In [ ]:
# target_speaker_embedding: 학습 중 추출한 화자 임베딩을 저장할 변수
# 나중에 체크포인트에 저장할 때 사용됩니다
target_speaker_embedding = None

# model.train(): 모델을 학습 모드로 전환
#   Talker LLM =  음소(phoneme) 정체성, 기본 피치 곡선(F0 trajectory), 음성/무음 구분
#                   ↓
#   code_predictor = 15명의 도우미
#   코드북 1~2   │ 화자 정체성 (목소리 timbre), 큰 운율(prosody), 말하는 속도 
#   코드북 3~5   │ 음성 품질 (breathiness, creakiness), 발음 정밀도, 감정 톤
#   코드북 6~9   │ 고주파 스펙트럴 디테일, 자음 명료도, 미세 운율 변동
#   코드북 10~12 │ 녹음 환경 특성 (방의 reverb, 마이크 색깔), 미세 노이즈 분포
#   코드북 13~15 │ 미세한 phase 정보, 인지 한계 근처의 polish
#                   ↓
#   16개 합쳐서 = 풍부한 합주
model.train()

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    for step, batch in enumerate(train_dataloader):
        # accelerator.accumulate(model): 그래디언트 누적을 자동 관리
        # GRAD_ACCUM_STEPS번 누적 후에만 실제로 optimizer.step()이 실행됨
        with accelerator.accumulate(model):

            # ============================================================
            # [7-1] 배치 데이터 언패킹
            # DataLoader에서 나온 배치 딕셔너리에서 각 텐서를 꺼냅니다
            # ============================================================
            input_ids = batch['input_ids']                    # (B, T, 2) 텍스트+코덱 토큰 ID
            codec_ids = batch['codec_ids']                    # (B, T, 16) 16개 코드북 정답
            ref_mels = batch['ref_mels']                      # (B, T_mel, 128) 참조 음성의 Mel
            text_embedding_mask = batch['text_embedding_mask'] # (B, T, 1) 텍스트 위치 마스크
            codec_embedding_mask = batch['codec_embedding_mask'] # (B, T, 1) 코덱 위치 마스크
            attention_mask = batch['attention_mask']           # (B, T) 어텐션 마스크
            codec_0_labels = batch['codec_0_labels']           # (B, T) 코드북0 정답 라벨
            codec_mask = batch['codec_mask']                   # (B, T) 코덱 영역 마스크

            # ============================================================
            # [7-2] Speaker Embedding 추출
            # 참조 음성의 Mel Spectrogram에서 화자의 "목소리 지문"을 추출합니다
            # .detach()로 Speaker Encoder의 가중치는 학습하지 않습니다 (고정)
            # ============================================================
            speaker_embedding = model.speaker_encoder(
                ref_mels.to(model.device).to(model.dtype)   # GPU로 이동 + 데이터 타입 맞춤
            ).detach()    # 이 연산은 역전파하지 않음 (Speaker Encoder 고정)
            # PyTorch에서 모든 텐서는 "내가 어떻게 계산되어 나왔는지"에 대한 족보(계산 그래프, computation graph)를 들고 다니는데, .detach()는 바로 이 족보를 끊어버립니다.

            # 첫 번째로 추출한 화자 임베딩을 저장 (나중에 체크포인트에 기록)
            if target_speaker_embedding is None:
                target_speaker_embedding = speaker_embedding

            # ============================================================
            # input_ids에서 텍스트 채널(0)과 코덱 채널(1)을 분리
            input_text_ids = input_ids[:, :, 0]    # (B, T) 텍스트 토큰 ID
            input_codec_ids = input_ids[:, :, 1]   # (B, T) 코덱 토큰 ID

            # 일반 LLM vs Qwen3-TTS 의 input_ids 차이
            #   표준 LLM (GPT, BERT, Llama 등)
            #   input_ids.shape == (B, T)        # (batch, sequence_length)
            #   # 한 위치에 토큰 하나
            #   position[0]: [한]    ← vocab index 정수 1개
            #   position[1]: [녕]
            #   position[2]: [하]
            #   → 매 위치마다 vocab(예: 32000개) 에서 정수 1개.
            
            #   Qwen3-TTS
            #   input_ids.shape == (B, T, 2)     # 마지막 차원 = 2 채널!
            #   # 한 위치에 토큰 두 개 — 같은 timestep 에 동시 존재
            #   position[0]: [한, 142]    ← [텍스트 토큰, 코덱 토큰] 동시에
            #   position[1]: [녕, 87]
            #   position[2]: [하, 523]
            #   → 매 위치마다 두 개의 vocabulary 에서 정수 하나씩. 그래서 [:, :, 0] 으로 텍스트만, [:, :, 1] 으로 코덱만 추출하는 거예요.
            # ============================================================


            # ============================================================
            # [7-3] Text Embedding 생성
            # 텍스트 토큰 ID를 연속 벡터(임베딩)로 변환합니다
            # 0.6B 모델은 차원을 맞추기 위해 text_projection을 추가 적용
            # ============================================================
            if needs_projection:
                # 0.6B: text_embedding(2048) -> text_projection -> (1024)
                input_text_embedding = model.talker.text_projection(
                    model.talker.model.text_embedding(input_text_ids)
                ) * text_embedding_mask
            else:
                # 1.7B: 차원이 같으므로 projection 불필요
                input_text_embedding = model.talker.model.text_embedding(
                    input_text_ids
                ) * text_embedding_mask

            # ============================================================
            # [7-4] Codec Embedding 생성 + 화자 임베딩 주입
            # 코덱 토큰 ID를 벡터로 변환하고, Position 6에 화자 정보를 넣습니다
            # ============================================================
            input_codec_embedding = model.talker.model.codec_embedding(
                input_codec_ids        # 코덱 토큰 ID -> 벡터
            ) * codec_embedding_mask   # 유효하지 않은 위치는 0으로 마스킹

            # Position 6에 화자 임베딩을 직접 주입!
            # 이 위치가 모델에게 "누구의 목소리로 말할지" 알려주는 슬롯입니다
            #   - 위치 6: 항상 화자 슬롯
            #   - 위치 0~5: 시스템/태스크 토큰
            #   - 위치 7 이후: 실제 텍스트/코덱 콘텐츠
            input_codec_embedding[:, 6, :] = speaker_embedding
              

            # ============================================================
            # [7-5] 임베딩 합산 (텍스트 + 코덱)
            # 두 채널의 임베딩을 요소별로 더합니다
            # ============================================================
            input_embeddings = input_text_embedding + input_codec_embedding

            # ============================================================
            # [7-6] Sub-codebook 임베딩 추가 (코드북 1~15)
            # 코드북 0 외에 나머지 15개 코드북의 정보를 순서대로 더합니다
            # 각 코드북은 별도의 임베딩 테이블을 가지고 있습니다
            # ============================================================
            for i in range(1, 16):
                # code_predictor의 i-1번째 임베딩 테이블에서 코드북 i의 벡터를 가져옴

                # 🧩 code_predictor 의 내부 구조

                # "하나의 네트워크" 라고는 했지만, 그 안에 코드북 1~15 를 처리하기 위한 15개의 입력 임베딩 테이블 + 15개의 출력 헤드 가 들어있습니다. 
                
                # code_predictor (네트워크 1개)
                # ├── 입력 임베딩 테이블들 (코드북별로 다른 vocabulary 처리)
                # │   ├── emb[0]: 코드북 1 의 임베딩 테이블
                # │   ├── emb[1]: 코드북 2 의 임베딩 테이블
                # │   ├── emb[2]: 코드북 3 의 임베딩 테이블
                # │   ├── ...
                # │   └── emb[14]: 코드북 15 의 임베딩 테이블
                # │
                # ├── 공통 layers (트랜스포머 / MLP 등)
                # │   ← 이 부분은 15개 코드북이 공유 사용
                # │
                # └── 출력 헤드들 (코드북별로 다른 vocab 으로 분류)
                #     ├── head[0]: 코드북 1 의 logits 출력
                #     ├── head[1]: 코드북 2 의 logits 출력
                #     ├── ...
                #     └── head[14]: 코드북 15 의 logits 출력

                codec_i_embedding = model.talker.code_predictor.get_input_embeddings()[i - 1](
                    codec_ids[:, :, i]     # i번째 코드북의 토큰 ID
                )
                # 코덱 토큰이 없는 영역은 0으로 마스킹
                codec_i_embedding = codec_i_embedding * codec_mask.unsqueeze(-1)
                # 기존 임베딩에 더함 (16개 코드북 정보가 모두 합쳐짐)
                input_embeddings = input_embeddings + codec_i_embedding

            # ============================================================
            # [7-7] Forward Pass (Talker LLM 실행)
            # 합쳐진 임베딩을 Talker에 넣어서 다음 코덱 토큰을 예측합니다
            # ============================================================
            outputs = model.talker(
                inputs_embeds=input_embeddings[:, :-1, :],  # 마지막 하나 빼고 입력 (다음 토큰 예측)
                attention_mask=attention_mask[:, :-1],       # 패딩 위치 무시
                labels=codec_0_labels[:, 1:],                # 한 칸 밀린 정답 (t입력 -> t+1예측)
                output_hidden_states=True                    # 히든 스테이트도 반환
            )

            # ============================================================
            # [7-8] Sub-talker Loss 계산
            # Talker의 히든 스테이트를 사용해 코드북 1~15의 예측 Loss를 계산합니다
            # ============================================================
            # Talker 마지막 레이어의 히든 스테이트를 가져옴
            hidden_states = outputs.hidden_states[0][-1]
            # 코덱 영역(음성이 있는 부분)의 히든 스테이트만 추출
            talker_hidden_states = hidden_states[codec_mask[:, :-1]]
            # 대응하는 정답 코덱 ID도 추출
            talker_codec_ids = codec_ids[codec_mask]

            # Sub-talker로 코드북 1~15 예측 및 Loss 계산
            sub_talker_logits, sub_talker_loss = model.talker.forward_sub_talker_finetune(
                talker_codec_ids, talker_hidden_states
            )

            # ============================================================
            # [7-9] 최종 Loss 합산 + Backward (역전파) + 가중치 업데이트
            # ============================================================
            # 최종 Loss = Talker Loss (코드북0) + 0.3 x Sub-talker Loss (코드북1~15)
            # 코드북 0이 더 중요하므로 비중을 높게 줌 (1.0 vs 0.3)
            loss = outputs.loss + 0.3 * sub_talker_loss

            # 역전파: Loss로부터 모든 가중치의 기울기(gradient)를 계산
            accelerator.backward(loss)

            # 기울기 클리핑: 기울기가 너무 크면 1.0으로 잘라냄 (학습 안정화)
            # 기울기가 폭발하면 가중치가 갑자기 크게 변해 학습이 망가질 수 있음
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), 1.0)

            # 옵티마이저: 계산된 기울기를 사용해 가중치를 업데이트
            optimizer.step()

            # 기울기 초기화: 다음 스텝의 기울기가 누적되지 않도록
            optimizer.zero_grad()

        # 10 스텝마다 현재 Loss를 출력 (학습 진행 상황 모니터링)
        if step % 10 == 0:
            print(f"  Step {step} | Loss: {loss.item():.4f}")

    print(f"\nEpoch {epoch} 완료!")
    # ====================================================================
    # epoch 완료 후: 이 시점 모델 상태를 체크포인트로 저장
    # (sft_12hz_compatible_temp.py 와 동일하게 매 epoch 저장 →
    #  epoch 별 가중치가 진짜로 달라지므로 over-fit 회피용 비교 가능)
    # ====================================================================
    if accelerator.is_main_process:
        # INIT_MODEL_PATH 가 HF Hub ID 면 로컬 캐시 경로로 해석
        if os.path.isdir(INIT_MODEL_PATH):
            _src = INIT_MODEL_PATH
        else:
            from huggingface_hub import snapshot_download
            _src = snapshot_download(repo_id=INIT_MODEL_PATH)

        output_dir = os.path.join(OUTPUT_MODEL_PATH, f"checkpoint-epoch-{epoch}")
        shutil.copytree(_src, output_dir, dirs_exist_ok=True)

        # config.json 수정: 커스텀 화자 정보 등록
        with open(os.path.join(_src, "config.json"), "r", encoding="utf-8") as f:
            config_dict = json.load(f)
        config_dict["tts_model_type"] = "custom_voice"
        tc = config_dict.get("talker_config", {})
        tc["spk_id"]         = {SPEAKER_NAME: 3000}
        tc["spk_is_dialect"] = {SPEAKER_NAME: False}
        config_dict["talker_config"] = tc
        with open(os.path.join(output_dir, "config.json"), "w", encoding="utf-8") as f:
            json.dump(config_dict, f, indent=2, ensure_ascii=False)

        # 모델 가중치 추출 → speaker_encoder 제거 → 화자 임베딩 주입
        unwrapped_model = accelerator.unwrap_model(model)
        state_dict = {k: v.detach().to("cpu") for k, v in unwrapped_model.state_dict().items()}
        for k in [k for k in state_dict.keys() if k.startswith("speaker_encoder")]:
            del state_dict[k]
        w = state_dict["talker.model.codec_embedding.weight"]
        w[3000] = target_speaker_embedding[0].detach().to(w.device).to(w.dtype)
        save_file(state_dict, os.path.join(output_dir, "model.safetensors"))

        print(f"  ✅ checkpoint-epoch-{epoch} 저장 완료 → {output_dir}")



Epoch 0/8
  Step 0 | Loss: 12.9764
  Step 10 | Loss: 14.3935
  Step 20 | Loss: 13.9672
  Step 30 | Loss: 14.0598
  Step 40 | Loss: 12.9900
  Step 50 | Loss: 14.1202

Epoch 0 완료!


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 81748.05it/s]


  ✅ checkpoint-epoch-0 저장 완료 → /dataset/research/temp/AI_workshop/chapter5-2_TTS/outputs/finetuned_korean_tts/checkpoint-epoch-0

Epoch 1/8
  Step 0 | Loss: 13.2564
  Step 10 | Loss: 13.2467
  Step 20 | Loss: 14.0068
  Step 30 | Loss: 13.0685
  Step 40 | Loss: 13.1614
  Step 50 | Loss: 11.8930

Epoch 1 완료!


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 82866.19it/s]


  ✅ checkpoint-epoch-1 저장 완료 → /dataset/research/temp/AI_workshop/chapter5-2_TTS/outputs/finetuned_korean_tts/checkpoint-epoch-1

Epoch 2/8
  Step 0 | Loss: 12.9489
  Step 10 | Loss: 13.0220
  Step 20 | Loss: 12.1133
  Step 30 | Loss: 13.3573
  Step 40 | Loss: 13.5979
  Step 50 | Loss: 12.4532

Epoch 2 완료!


...

  ✅ checkpoint-epoch-5 저장 완료 → /dataset/research/temp/AI_workshop/chapter5-2_TTS/outputs/finetuned_korean_tts/checkpoint-epoch-5

Epoch 6/8
  Step 0 | Loss: 14.0067
  Step 10 | Loss: 13.0787
  Step 20 | Loss: 13.2674
  Step 30 | Loss: 13.6886
  Step 40 | Loss: 14.4792
  Step 50 | Loss: 13.7061

Epoch 6 완료!


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 31775.03it/s]


  ✅ checkpoint-epoch-6 저장 완료 → /dataset/research/temp/AI_workshop/chapter5-2_TTS/outputs/finetuned_korean_tts/checkpoint-epoch-6

Epoch 7/8
  Step 0 | Loss: 12.4259
  Step 10 | Loss: 13.8596
  Step 20 | Loss: 13.2769
  Step 30 | Loss: 12.2825
  Step 40 | Loss: 14.2378
  Step 50 | Loss: 12.3087

Epoch 7 완료!


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 146575.14it/s]


  ✅ checkpoint-epoch-7 저장 완료 → /dataset/research/temp/AI_workshop/chapter5-2_TTS/outputs/finetuned_korean_tts/checkpoint-epoch-7


## 8단계: 체크포인트 저장

학습이 완료된 모델을 파일로 저장합니다. 이 과정에서 몇 가지 중요한 작업이 이루어집니다.

**원본 폴더 복사**: Qwen3-TTS 모델을 로드하려면 가중치 파일 외에도 config, tokenizer 등 여러 파일이 필요하므로, 원본 폴더를 통째로 복사한 뒤 변경된 부분만 덮어씁니다.

**config.json 수정**: `tts_model_type`을 `"custom_voice"`로 변경하여 추론 시 모델이 커스텀 화자 모델임을 인식하게 합니다. 또한 `spk_id`에 화자 이름과 코덱 임베딩 인덱스의 매핑을 등록합니다.

**Speaker Embedding 저장**: 학습 중 추출한 화자 임베딩을 `codec_embedding.weight[3000]`에 기록합니다. 코덱 임베딩 테이블은 3072개의 행을 가진 큰 테이블인데, 인덱스 3000번 행에 해당 화자의 목소리 지문을 저장하는 것입니다. 추론 시에는 화자 이름으로 이 인덱스를 찾아 임베딩을 꺼내 사용합니다.

**Speaker Encoder 제거**: 저장하는 모델에서 `speaker_encoder` 가중치는 제거합니다. 추론 시에는 참조 음성 대신 저장된 화자 임베딩을 직접 사용하므로, Speaker Encoder가 필요 없기 때문입니다.


In [14]:
# ============================================================
# 저장된 체크포인트 목록 확인
# ============================================================
# (이전 버전에서는 이 셀에서 for epoch in range(NUM_EPOCHS) 루프로
#  체크포인트를 저장했는데, 모든 폴더에 "학습 끝난 최종 모델"이
#  복제되는 버그가 있었습니다.
#  지금은 Cell 19 의 학습 루프 안에서 매 epoch 별로 저장하므로
#  각 폴더의 가중치가 진짜로 다릅니다.)

checkpoints = sorted(
    [d for d in os.listdir(OUTPUT_MODEL_PATH) if d.startswith("checkpoint-epoch-")],
    key=lambda s: int(s.rsplit("-", 1)[-1])
)

print(f"저장된 체크포인트 ({len(checkpoints)}개):")
for ckpt in checkpoints:
    safetensor = os.path.join(OUTPUT_MODEL_PATH, ckpt, "model.safetensors")
    size_mb = os.path.getsize(safetensor) / 1024**2 if os.path.exists(safetensor) else 0
    print(f"  - {ckpt:25s}  model.safetensors {size_mb:6.1f} MB")


저장된 체크포인트 (8개):
  - checkpoint-epoch-0         model.safetensors 1727.7 MB
  - checkpoint-epoch-1         model.safetensors 1727.7 MB
  - checkpoint-epoch-2         model.safetensors 1727.7 MB
  - checkpoint-epoch-3         model.safetensors 1727.7 MB
  - checkpoint-epoch-4         model.safetensors 1727.7 MB
  - checkpoint-epoch-5         model.safetensors 1727.7 MB
  - checkpoint-epoch-6         model.safetensors 1727.7 MB
  - checkpoint-epoch-7         model.safetensors 1727.7 MB


## 9단계: 인퍼런스 테스트

학습이 잘 되었는지 확인하기 위해, Fine-tuning된 모델로 음성을 생성해봅니다. Jupyter Notebook에서 바로 재생할 수 있습니다.

추론 과정은 다음과 같습니다. 먼저 config에서 화자 이름으로 `spk_id` 인덱스를 찾고, `codec_embedding.weight`의 해당 행에서 화자 임베딩을 꺼냅니다. 그 다음 텍스트와 화자 정보를 결합하여 코덱 토큰을 생성하고, 이 토큰을 음성 파형으로 디코딩하면 최종 음성이 만들어집니다.


In [ ]:
import soundfile as sf                       # 음성 파일 읽기/쓰기
from IPython.display import Audio, display   # Jupyter에서 오디오 재생

# 학습된 체크포인트 경로
# checkpoint-epoch-N 은 (N+1) epoch 학습이 끝난 시점의 모델입니다.
# 즉 NUM_EPOCHS=8 로 학습하면 checkpoint-epoch-0 ~ checkpoint-epoch-7 까지 생성됩니다.
# 본 노트북 작성자는 마지막 epoch인 checkpoint-epoch-7 로 테스트했을 때
# 30개 한국어 문장 모두 정상 생성됨을 확인했습니다.
CHECKPOINT = os.path.join(OUTPUT_MODEL_PATH, "checkpoint-epoch-7")
DEVICE = "cuda:0"

# Fine-tuning된 모델 로드
# 이 모델은 학습된 화자 임베딩이 포함되어 있어서
# speaker 이름만으로 해당 화자의 목소리로 음성을 생성할 수 있습니다
test_model = Qwen3TTSModel.from_pretrained(
    CHECKPOINT,
    device_map=DEVICE,
    dtype=torch.bfloat16,
)

print(f"테스트 모델 로드 완료: {CHECKPOINT}")


### 음성 생성 및 재생

아래 셀에서 텍스트를 바꿔가며 테스트해보세요. 확인할 점은 발음의 정확성, 원래 화자와의 음색 유사도, 억양과 리듬의 자연스러움, 잡음이나 깨짐 여부입니다.


In [16]:
# 테스트할 텍스트 목록 (자유롭게 수정하세요!)
test_texts = [
    "안녕하세요, 반갑습니다.",
    "오늘 날씨가 정말 좋네요.",
    "이 제품은 품질이 우수합니다.",
]

for i, text in enumerate(test_texts):
    print(f"\n{'─'*40}")
    print(f"[{i+1}] 입력 텍스트: {text}")
    print(f"{'─'*40}")

    # generate_custom_voice: 커스텀 화자의 목소리로 음성 생성
    # - text: 읽을 텍스트
    # - speaker: config.json에 등록된 화자 이름
    wavs, sr = test_model.generate_custom_voice(
        text=text,
        speaker=SPEAKER_NAME,
    )

    # wavs[0]: 생성된 음성 파형 (numpy array)
    # sr: 샘플링 레이트 (보통 24000Hz)
    duration = len(wavs[0]) / sr
    print(f"    생성된 음성 길이: {duration:.2f}초")
    print(f"    샘플링 레이트: {sr}Hz")
    print(f"    샘플 수: {len(wavs[0])}")

    # IPython.display.Audio로 노트북에서 바로 재생
    display(Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



────────────────────────────────────────
[1] 입력 텍스트: 안녕하세요, 반갑습니다.
────────────────────────────────────────
    생성된 음성 길이: 2.14초
    샘플링 레이트: 24000Hz
    샘플 수: 51285


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



────────────────────────────────────────
[2] 입력 텍스트: 오늘 날씨가 정말 좋네요.
────────────────────────────────────────
    생성된 음성 길이: 2.70초
    샘플링 레이트: 24000Hz
    샘플 수: 64725


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



────────────────────────────────────────
[3] 입력 텍스트: 이 제품은 품질이 우수합니다.
────────────────────────────────────────
    생성된 음성 길이: 2.78초
    샘플링 레이트: 24000Hz
    샘플 수: 66645


## 음성을 파일로 저장하고 싶다면

위에서 생성한 음성을 WAV 파일로 저장할 수 있습니다.


In [ ]:
# 음성 파일로 저장하기 (필요한 경우 실행)
SAVE_DIR = os.path.join(OUTPUT_MODEL_PATH, "test_outputs")
os.makedirs(SAVE_DIR, exist_ok=True)

for i, text in enumerate(test_texts):
    wavs, sr = test_model.generate_custom_voice(
        text=text,
        speaker=SPEAKER_NAME,
    )
    save_path = os.path.join(SAVE_DIR, f"test_{i+1}.wav")
    sf.write(save_path, wavs[0], sr)
    print(f"저장 완료: {save_path}")

print(f"\n모든 테스트 음성이 {SAVE_DIR}에 저장되었습니다.")

### 🔍 최종 검증 — 학습 전 ↔ 학습 데이터 원본 비교

Fine-tuning 결과는 이미 위 Cell 28-29 에서 들으셨습니다. 여기서는 비교 기준이 되는 두 가지를 다시 재생해, **직접 청취 비교** 할 수 있게 합니다:

| 청취 | 의미 |
|------|------|
| 위 Cell 28-29 | **학습 후 Fine-tuned 결과** — 이게 [1] 과 비슷하면 성공 |
| **[1] 학습 데이터 원본** | 목표 화자의 진짜 음성 (정답지) |
| **[2] 학습 전 sohee** | Cell 17 에서 저장된 학습 전 베이스라인 |

세 가지를 모두 같은 문장으로 비교하므로, 변수는 오직 **모델/화자** 뿐입니다.

In [18]:
# ============================================================
# 학습 전 ↔ 학습 데이터 원본 비교
# ============================================================
# [학습 후 결과] 는 바로 위 Cell 28-29 에서 이미 들으셨습니다.
# 여기서는 그 결과와 비교할 두 가지 — 원본 / 학습 전 — 만 재생합니다:
#   [1] 학습 데이터 원본       (목표 화자가 진짜로 말한 음성)
#   [2] 학습 전 sohee 발성     (Cell 17 에서 저장된 결과)
# 위 Cell 28-29 의 학습 후 결과가 [1] 에 가까우면 성공, [2] 에 가까우면 학습 부족.
# ============================================================
from IPython.display import Audio, display
import json, os

raw_jsonl = os.path.join(DATA_DIR, "train_raw.jsonl")
with open(raw_jsonl) as f:
    first_sample = json.loads(f.readlines()[0])
compare_text = first_sample["text"]   # [1], [2] 모두 이 문장

print("=" * 60)
print("  학습 전 ↔ 학습 데이터 원본  (같은 문장으로 비교)")
print("=" * 60)
print(f"  비교 문장: {compare_text}")
print("=" * 60)

# ---- [1] 학습 데이터 원본 (목표 화자) ----
print(f"\n[1] 학습 데이터 원본 — 목표 화자의 실제 녹음")
display(Audio(first_sample["audio"], rate=24000))

# ---- [2] 학습 전 sohee (Cell 17 결과) ----
base_audio_path = os.path.join(BASE_DIR, "_base_model_korean_test.wav")
print(f"\n[2] 학습 전 — Qwen sohee 화자 (CustomVoice 사전등록 한국어 여성)")
if os.path.exists(base_audio_path):
    display(Audio(base_audio_path))
else:
    print(f"    (Cell 17 이 실행되지 않았거나 실패했습니다 — 생략)")

print("\n" + "=" * 60)
print("  체크 포인트 — 위 Cell 28-29 의 학습 후 결과와 비교:")
print("    ✓ 학습 후 결과가 [1] 과 비슷 → 화자 클로닝 성공!")
print("    ✗ 학습 후 결과가 여전히 [2] 와 비슷 → 학습 부족 (epoch ↑ 또는 LR ↑)")
print("    ✗ 학습 후 결과가 너무 길거나 발음이 깨짐 → over-fit (LR ↓)")
print("=" * 60)


  학습 전 ↔ 학습 데이터 원본  (같은 문장으로 비교)
  비교 문장: 고객센터 번호는 1588-3920이고, 팩스는 02-555-7812입니다.

[1] 학습 데이터 원본 — 목표 화자의 실제 녹음



[2] 학습 전 — Qwen sohee 화자 (CustomVoice 사전등록 한국어 여성)



  체크 포인트 — 위 Cell 28-29 의 학습 후 결과와 비교:
    ✓ 학습 후 결과가 [1] 과 비슷 → 화자 클로닝 성공!
    ✗ 학습 후 결과가 여전히 [2] 와 비슷 → 학습 부족 (epoch ↑ 또는 LR ↑)
    ✗ 학습 후 결과가 너무 길거나 발음이 깨짐 → over-fit (LR ↓)


## 자주 묻는 질문

##### Q1. Loss가 얼마나 내려가야 좋은 건가요?

절대적인 기준은 없지만, 일반적으로 학습 시작 시 14 정도이던 Loss가 13 또는 7 정도까지 내려가면 양호한 수준입니다. 다만 0에 너무 가까워지면 과적합을 의심해야 합니다.

#### Q2. Epoch 수는 어떻게 정하나요?

데이터가 적으면(10~50개) 5~10 에폭, 보통이면(50~200개) 3~5 에폭, 많으면(500개 이상) 1~3 에폭이 적당합니다. 음색이 이상해지거나 부자연스러워지면 과적합이므로 에폭 수를 줄여야 합니다.

#### Q3. 0.6B와 1.7B 중 어떤 모델을 선택하나요?

GPU 메모리가 8~16GB이거나 빠른 실험이 필요하면 0.6B, 24GB 이상이고 최상의 품질을 원하면 1.7B를 선택합니다. 0.6B로 먼저 실험한 뒤 검증이 끝나면 1.7B로 전환하는 것을 권장합니다.

#### Q4. 다중 화자를 한 모델에서 학습할 수 있나요?

가능합니다. 각 화자에게 서로 다른 `codec_embedding` 인덱스(예: 3000, 3001, 3002)를 할당하고, 학습 데이터에 `speaker_id` 필드를 추가하면 됩니다. 자세한 구현 방법은 `multi-speaker-sft` 문서를 참고하세요.
